INSTALL ALL REQUIRED LIBRARIES

In [ ]:

!pip install -q sentence-transformers scikit-learn umap-learn plotly

import pandas as pd
import numpy as np
import re
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report
import umap
import plotly.express as px

LOAD DATASET

In [ ]:

df = pd. read_csv('/content/clinical_notes_diagnosis_prediction_5000.csv')

DATA CLEANING AND EDA

In [ ]:
df

,Clinical Notes,Diagnosis
0,"A 35-year-old male presents with heartburn, re...",Gastroesophageal Reflux Disease
1,A 55-year-old male with a history of heavy alc...,Liver Cirrhosis
2,A 50-year-old male with a history of diabetes ...,Septic Shock
3,"A 35-year-old male presents with heartburn, re...",Gastroesophageal Reflux Disease
4,A 55-year-old female with a history of obesity...,Type 2 Diabetes Mellitus
...,...,...
4995,A 55-year-old male with a history of heavy alc...,Liver Cirrhosis
4996,"A 40-year-old female presents with swelling, p...",Deep Vein Thrombosis
4997,A 55-year-old male with a history of heavy alc...,Liver Cirrhosis
4998,"A 40-year-old female presents with swelling, p...",Deep Vein Thrombosis


In [ ]:
df. shape

(5000, 2)

In [ ]:

df.columns.tolist()

['Clinical Notes', 'Diagnosis']

In [ ]:
df[['Clinical Notes', 'Diagnosis']].head(3)

,Clinical Notes,Diagnosis
0,"A 35-year-old male presents with heartburn, re...",Gastroesophageal Reflux Disease
1,A 55-year-old male with a history of heavy alc...,Liver Cirrhosis
2,A 50-year-old male with a history of diabetes ...,Septic Shock


In [ ]:
def clean_clinical_note(text):
    text = str(text).lower().strip()
    text = re.sub(r'[^a-z0-9\s/.,mgdL%-]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text

df['clean_note'] = df['Clinical Notes'].apply(clean_clinical_note)

df = df.dropna(subset=['clean_note', 'Diagnosis']).reset_index(drop=True)

print("Cleaned sample note:")
print(df['clean_note'].iloc[0])

Cleaned sample note:
a 35-year-old male presents with heartburn, regurgitation, and a sour taste in his mouth, especially after meals. the patient has been self-medicating with over-the-counter antacids, but the symptoms persist. a 24-hour ph monitoring test confirms the diagnosis of gerd. the patient is started on a ppi and advised to avoid trigger foods.


GENERATE EMBEDDING FOR ALL CLINICAL NOTES

In [ ]:

model = SentenceTransformer('all-MiniLM-L6-v2')


note_embeddings = model.encode(
    df['clean_note'].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True,
    batch_size=30
)

In [ ]:
note_embeddings.shape

(5000, 384)


     COMPUTE SIMILARITY AGAINST ALL HISTORICAL NOTES

In [ ]:
def get_diagnosis_suggestion(new_note, top_k=3, similarity_threshold=0.6):
    """
    Input: New raw clinical note
    Output: Top similar past notes + their diagnoses + similarity scores
    """
    query_clean = clean_clinical_note(new_note)
    query_emb = model.encode([query_clean], normalize_embeddings=True)

    sims = cosine_similarity(query_emb, note_embeddings)[0]
    top_idx = np.argsort(sims)[::-1][:top_k]

    results = []
    for idx in top_idx:
        score = sims[idx]
        if score >= similarity_threshold:
            results.append({
                'matched_note': df.iloc[idx]['clinical_note'],
                'diagnosis': df.iloc[idx]['diagnosis'],
                'similarity_to_query': round(float(score), 4)
            })

    if not results:
        return {
            'status': 'No sufficiently similar past cases found',
            'best_match': df.iloc[np.argmax(sims)]['clinical_note'],
            'best_score': round(float(sims.max()), 4)
        }

    return {'status': f'Found {len(results)} similar cases', 'matches': results}

COMPUTER PAIRWISE SIMILARITY BETWEEN THE TOP-K CANDIDATES

In [ ]:
def find_similar_note_pairs(new_note, top_k=5, pair_similarity_threshold=0.8):
    """
    Returns:
    - For each similar pair: score of A vs query, score of B vs query, similarity between A and B
    """
    query_clean = clean_clinical_note(new_note)
    query_emb = model.encode([query_clean], normalize_embeddings=True)
    sims_to_query = cosine_similarity(query_emb, note_embeddings)[0]
    top_idx = np.argsort(sims_to_query)[::-1][:top_k]


    sub_embeddings = note_embeddings[top_idx]
    pairwise_sim = cosine_similarity(sub_embeddings)

    similar_pairs = []
    for i in range(len(top_idx)):
        for j in range(i+1, len(top_idx)):
            pair_score = pairwise_sim[i,j]
            if pair_score >= pair_similarity_threshold:
                idx_a, idx_b = top_idx[i], top_idx[j]
                similar_pairs.append({
                    'note_A': {
                        'diagnosis': df.iloc[idx_a]['diagnosis'],
                        'note_text': df.iloc[idx_a]['clinical_note'],
                        'score_vs_query': round(float(sims_to_query[idx_a]), 4)
                    },
                    'note_B': {
                        'diagnosis': df.iloc[idx_b]['diagnosis'],
                        'note_text': df.iloc[idx_b]['clinical_note'],
                        'score_vs_query': round(float(sims_to_query[idx_b]), 4)
                    },
                    'similarity_between_A_and_B': round(float(pair_score), 4)
                })

    if not similar_pairs:
        return {'status': 'No highly similar pairs found among top matches'}
    return {'status': f'Found {len(similar_pairs)} similar pair(s)', 'pairs': similar_pairs}

In [ ]:
def analyze_clinical_case(new_note, top_k=5):
    print("="*80)
    print("NEW CLINICAL NOTE INPUT:")
    print(new_note)
    print("="*80)


    suggestion_output = get_diagnosis_suggestion(new_note, top_k=top_k)
    print("\\n🔹 DIAGNOSIS SUGGESTIONS FROM SIMILAR CASES:")
    print(suggestion_output['status'])
    for m in suggestion_output.get('matches', []):
        print(f" • Score: {m['similarity_to_query']} | Diagnosis: {m['diagnosis']}")


    pair_output = find_similar_note_pairs(new_note, top_k=top_k)
    print("\\n🔹 SIMILAR PAIRS (Both Scores + Mutual Similarity):")
    print(pair_output['status'])
    for p in pair_output.get('pairs', []):
        print(f" • A (score={p['note_A']['score_vs_query']}, dx={p['note_A']['diagnosis']}) ↔ B (score={p['note_B']['score_vs_query']}, dx={p['note_B']['diagnosis']}) | A↔B sim={p['similarity_between_A_and_B']}")

In [ ]:
def semantic_clinical_search(query, top_k=5):
    query_emb = model.encode([clean_clinical_note(query)], normalize_embeddings=True)
    sims = cosine_similarity(query_emb, note_embeddings)[0]
    top_idx = np.argsort(sims)[::-1][:top_k]

    results = []
    for idx in top_idx:
        results.append({
            'Diagnosis': df.iloc[idx]['Diagnosis'],
            'note': df.iloc[idx]['Clinical Notes'],
            'similarity': round(float(sims[idx]), 4)
        })
    return results

EXAMPLE

In [ ]:
semantic_clinical_search("asthma exacerbation wheezing shortness of breath", top_k=2)

[{'Diagnosis': 'Chronic Obstructive Pulmonary Disease',
  'note': 'A 70-year-old male with a long history of smoking presents with a productive cough and shortness of breath. He reports worsening symptoms over the past year, particularly in the mornings. On examination, wheezing is noted, and lung auscultation reveals decreased breath sounds. A chest X-ray reveals hyperinflation of the lungs. Pulmonary function tests confirm the diagnosis of COPD. The patient is started on bronchodilators and corticosteroids.',
  'similarity': 0.6546},
 {'Diagnosis': 'Chronic Obstructive Pulmonary Disease',
  'note': 'A 70-year-old male with a long history of smoking presents with a productive cough and shortness of breath. He reports worsening symptoms over the past year, particularly in the mornings. On examination, wheezing is noted, and lung auscultation reveals decreased breath sounds. A chest X-ray reveals hyperinflation of the lungs. Pulmonary function tests confirm the diagnosis of COPD. The pa

USING LABEL ENCODER

In [ ]:
le = LabelEncoder()
df['diagnosis_enc'] = le.fit_transform(df['Diagnosis'])
num_classes = len(le.classes_)
print("Diagnosis classes:", le.classes_)


X_train, X_test, y_train, y_test = train_test_split(note_embeddings, df['diagnosis_enc'], test_size=0.2, random_state=42, stratify=df['diagnosis_enc'])


knn_clf = KNeighborsClassifier(n_neighbors=5, metric='cosine')
knn_clf.fit(X_train, y_train)
y_pred = knn_clf.predict(X_test)

print("\\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report

In [ ]:
report = classification_report(y_test, y_pred, target_names=le.classes_, output_dict=True)

In [ ]:
report_df = pd.DataFrame(report).transpose()

In [ ]:
report_df = report_df.drop(columns=['support'])

In [ ]:

class_metrics_df = report_df.drop(index=['accuracy', 'macro avg', 'weighted avg'], errors='ignore')

In [ ]:
class_metrics_df = class_metrics_df.astype(float)

PLOTTING HEATMAP

In [ ]:
global heatmap_fig
heatmap_fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(class_metrics_df, annot=True, cmap='viridis', fmt=".2f", linewidths=.5, cbar_kws={'label': 'Score'}, ax=ax)
plt.title('Classification Report Heatmap (Precision, Recall, F1-Score per Class)')
plt.xlabel('Metrics')
plt.ylabel('Diagnosis Class')
plt.tight_layout()
plt.show()

UMAP WITH 2D VISUALISATION

In [ ]:
reducer = umap.UMAP(metric='cosine', random_state=42)
embedding_2d = reducer.fit_transform(note_embeddings)

df['umap_x'] = embedding_2d[:,0]
df['umap_y'] = embedding_2d[:,1]

fig = px.scatter(df, x='umap_x', y='umap_y', color='Diagnosis', hover_data=['Clinical Notes'],
                 title='Clinical Note Embeddings by Diagnosis')
fig.show()

In [ ]:
def extract_age(note):
    match = re.search(r'(\d+)-year-old', note)
    if match:
        return int(match.group(1))
    return None

def extract_gender(note):
    if 'male' in note.lower():
        return 'Male'
    elif 'female' in note.lower():
        return 'Female'
    return None

df['age'] = df['Clinical Notes'].apply(extract_age)
df['gender'] = df['Clinical Notes'].apply(extract_gender)
display(df.head())

In [ ]:
print("\nGender distribution:")
display(df['gender'].value_counts())


Gender distribution:


,count
gender,
Male,5000


PREPROCESS NUMERIC DEMOGRAPHICS

In [ ]:

scaler = StandardScaler()
df['age_scaled'] = scaler.fit_transform(df[['age']])
df['gender_enc'] = (df['gender'] == 'Male').astype(int)

X_demo = df[['age_scaled', 'gender_enc']].values
X_demo_train, X_demo_test = train_test_split(X_demo, test_size=0.2, random_state=42)

BUILD HYBRID KERAS MODEL

In [ ]:

from tensorflow.keras.layers import Input, Dense, Dropout, Concatenate
from tensorflow.keras.models import Model

note_input = Input(shape=(note_embeddings.shape[1],))
t = Dense(64, activation='relu')(note_input)
t = Dropout(0.3)(t)

demo_input = Input(shape=(X_demo.shape[1],))
n = Dense(16, activation='relu')(demo_input)

combined = Concatenate()([t, n])
z = Dense(32, activation='relu')(combined)
output = Dense(num_classes, activation='softmax')(z)

hybrid_model = Model(inputs=[note_input, demo_input], outputs=output)
hybrid_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [ ]:
hybrid_model.fit([X_train, X_demo_train], y_train, validation_data=([X_test, X_demo_test], y_test), epochs=15, batch_size=32)

PREDICTING NEW NOTE AND GIVE SUGGESTIONS

In [ ]:
def predict_new_note(note_text):

    clean_note = clean_clinical_note(note_text)
    age = extract_age(note_text)
    gender = extract_gender(note_text)


    if age is not None:
        age_scaled = scaler.transform(np.array([[age]]))[0][0]
    else:

        age_scaled = scaler.transform(np.array([[df['age'].mean()]]))[0][0]


    gender_enc = 1 if gender == 'Male' else 0

    demo_features = np.array([[age_scaled, gender_enc]])


    note_embedding = model.encode([clean_note], normalize_embeddings=True)


    predictions = hybrid_model.predict([note_embedding, demo_features])
    predicted_class_index = np.argmax(predictions, axis=1)[0]
    predicted_diagnosis = le.inverse_transform([predicted_class_index])[0]

    return predicted_diagnosis, predictions

EXAMPLE

In [ ]:
example_note = "A 72-year-old female presents with progressive shortness of breath, bilateral lower extremity edema, and a history of hypertension. ECG shows left ventricular hypertrophy. Suspected congestive heart failure."
predicted_dx, probs = predict_new_note(example_note)

print(f"New Clinical Note: {example_note}")
print(f"Predicted Diagnosis: {predicted_dx}")
print(f"Prediction Probabilities: {probs}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
New Clinical Note: A 72-year-old female presents with progressive shortness of breath, bilateral lower extremity edema, and a history of hypertension. ECG shows left ventricular hypertrophy. Suspected congestive heart failure.
Predicted Diagnosis: Congestive Heart Failure
Prediction Probabilities: [[6.10623555e-03 5.07173445e-06 9.78451781e-03 1.31866639e-03
  2.29921311e-01 2.10139387e-05 2.29577608e-02 7.26413906e-01
  3.14050121e-04 5.37848799e-04 4.98763926e-04 3.71210888e-07
  5.77246654e-04 7.64730066e-05 9.09710536e-04 3.69073168e-05
  1.78158025e-05 1.03191505e-05 4.68282640e-04 2.38518896e-05]]


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning:

X does not have valid feature names, but StandardScaler was fitted with feature names



VISUALISATION OF PREDICTION

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
diagnosis_labels = le.classes_

In [ ]:
prob_df = pd.DataFrame({
    'Diagnosis': diagnosis_labels,
    'Probability': probs[0]
})

In [ ]:
prob_df = prob_df.sort_values(by='Probability', ascending=False)

In [93]:
top_n = 5
prob_df_top_n = prob_df.head(top_n)
display(prob_df_top_n)

,Diagnosis,Probability
7,Congestive Heart Failure,0.726414
4,Chronic Kidney Disease,0.229921
6,Community-Acquired Pneumonia,0.022958
2,Asthma,0.009785
0,Acute Myocardial Infarction,0.006106


PLOT THE DISTRIBUTION

In [ ]:
fig = plt.figure(figsize=(12, 8))
sns.barplot(x='Probability', y='Diagnosis', data=prob_df, palette='viridis')
plt.title('Probability Distribution of Predicted Diagnoses')
plt.xlabel('Probability')
plt.ylabel('Diagnosis')
plt.tight_layout()
plt.show()

TOP 5 DIAGNOSIS FOR FOCUSED VISUALISATION

BAR CHART WITH COLOUR MAPPING

In [ ]:
import plotly.express as px

In [ ]:

fig_bar_colormap = px.bar(prob_df.head(5),
                         x='Probability',
                         y='Diagnosis',
                         orientation='h',
                         color='Probability',
                         color_continuous_scale=['pink', 'darkred'],
                         title='Top 5 Predicted Diagnoses (Color Map)',
                         labels={'Probability': 'Probability', 'Diagnosis': 'Diagnosis'})

In [ ]:
fig_bar_colormap.update_layout(yaxis={'categoryorder':'total ascending'})
fig_bar_colormap.update_xaxes(tickvals=np.arange(0, 1.01, 0.15))
fig_bar_colormap.show()